# Faster R-CNN — Exploración Interactiva

Este notebook permite explorar el detector de forma interactiva:
- Detección en imágenes propias o de ejemplo
- Comparativa de variantes (v1, v2, Mask R-CNN)
- Análisis de arquitectura interna
- Fine-tuning paso a paso

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import urllib.request
from pathlib import Path

from src.detector import FasterRCNNDetector, COCO_CLASSES
from utils.visualization import visualize_detections, compare_models, detection_summary

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
print(f'Dispositivo: {"CUDA" if torch.cuda.is_available() else "CPU"}')

## 1. Cargar imagen de prueba

In [ ]:
# Descargar imagen de prueba
img_url  = 'https://ultralytics.com/images/zidane.jpg'
img_path = '../data/images/test.jpg'
Path('../data/images').mkdir(parents=True, exist_ok=True)

if not Path(img_path).exists():
    urllib.request.urlretrieve(img_url, img_path)
    print('Imagen descargada ✓')

img = cv2.imread(img_path)
plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f'Imagen de entrada  {img.shape[1]}×{img.shape[0]} px')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Detección con Faster R-CNN v2

In [ ]:
# Cargar modelo (descarga pesos ~160 MB la primera vez)
detector = FasterRCNNDetector(
    model_variant='v2',
    confidence_threshold=0.5,
)

# Detectar
result = detector.detect(img)

print(f'Objetos detectados : {len(result["boxes"])}')
print(f'Tiempo inferencia  : {result["time_ms"]:.1f} ms')
print(f'Clases detectadas  : {set(result["class_names"])}')

In [ ]:
# Visualizar con matplotlib
fig = visualize_detections(
    img,
    result['boxes'], result['scores'],
    result['labels'], COCO_CLASSES,
    figsize=(12, 7),
    show=True,
)

## 3. Análisis de la arquitectura interna

In [ ]:
model = detector.model

print('BACKBONE (Feature Extractor):')
print('  Tipo :', type(model.backbone).__name__)
print()
print('RPN (Region Proposal Network):')
print('  anchor_sizes  :', model.rpn.anchor_generator.sizes)
print('  aspect_ratios :', model.rpn.anchor_generator.aspect_ratios)
print('  nms_thresh    :', model.rpn.nms_thresh)
print()
print('ROI HEADS:')
print('  score_thresh  :', model.roi_heads.score_thresh)
print('  nms_thresh    :', model.roi_heads.nms_thresh)
print('  detections_per_img:', model.roi_heads.detections_per_img)
print()
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parámetros  : {total_params:,} ({total_params/1e6:.1f}M)')

## 4. Comparativa de variantes (v1 vs v2 vs Mask R-CNN)

In [ ]:
import time

variants = [
    ('v1',   'Faster R-CNN v1'),
    ('v2',   'Faster R-CNN v2'),
    ('mask', 'Mask R-CNN'),
]

compare_results = {}
for vid, vname in variants:
    det   = FasterRCNNDetector(vid, confidence_threshold=0.5)
    t0    = time.perf_counter()
    res   = det.detect(img)
    t_ms  = (time.perf_counter() - t0)*1000
    ann   = det.draw(img, res)
    compare_results[vname] = {'annotated': ann, 'time_ms': t_ms, 'n': len(res['boxes'])}
    print(f'{vname:<22}  objs={len(res["boxes"])}  t={t_ms:.0f}ms')

compare_models(img, compare_results, figsize=(18,6))

## 5. Ajuste de umbral de confianza

In [ ]:
thresholds = [0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, thresh in zip(axes, thresholds):
    det = FasterRCNNDetector('v2', confidence_threshold=thresh)
    res = det.detect(img)
    ann = det.draw(img, res)
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'conf ≥ {thresh}  ({len(res["boxes"])} objs)', fontsize=10)
    ax.axis('off')

plt.suptitle('Efecto del umbral de confianza', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Fine-tuning en dataset personalizado (esquema)

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Número de clases propias + fondo
NUM_CLASSES = 5 + 1  # ej: 5 clases tuyas + background

# Cargar modelo pre-entrenado
model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1)

# Reemplazar la cabeza de clasificación
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# Congelar backbone (transfer learning)
for name, param in model.named_parameters():
    if 'backbone' in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables : {trainable:,} ({100*trainable/total:.1f}%)')
print(f'Parámetros congelados  : {total-trainable:,}')
print(f'\nModelo listo para fine-tuning con {NUM_CLASSES} clases.')
print('Consulta src/train.py para el loop de entrenamiento completo.')